# Scenario Inputs & Model Comparison

This notebook demonstrates three related features:

1. **`InputLine` / `InputAssumption`** — mark model knobs explicitly so callers must supply them
2. **`hardcoded_color`** — visually distinguish fixed/input values in tables
3. **`ModelComparison`** — compare two or more models side-by-side

In [ ]:
from pyproforma.v2 import (
    Assumption,
    FixedLine,
    FormulaLine,
    InputAssumption,
    InputLine,
    ModelComparison,
    ProformaModel,
)

## 1. InputLine and InputAssumption

With `FixedLine` and `Assumption`, values are baked into the class definition and cannot be changed at instantiation time. `InputLine` and `InputAssumption` flip that: they declare a *knob* that the caller **must** (or optionally may) supply when creating the model instance.

This makes scenario analysis explicit and safe — the class definition documents exactly what can vary.

In [ ]:
class RetailModel(ProformaModel):
    """A simple retail P&L model with explicit scenario inputs."""

    # Fixed assumptions — same in every scenario
    cogs_ratio = Assumption(value=0.45, label="COGS Ratio")

    # Scenario inputs — caller must supply these at instantiation
    revenue = InputLine(label="Revenue")
    opex_ratio = InputAssumption(label="Operating Expense Ratio")

    # Calculated lines
    cogs = FormulaLine(
        formula=lambda li, t: li.revenue[t] * li.cogs_ratio,
        label="Cost of Goods Sold",
    )
    gross_profit = FormulaLine(
        formula=lambda li, t: li.revenue[t] - li.cogs[t],
        label="Gross Profit",
    )
    opex = FormulaLine(
        formula=lambda li, t: li.revenue[t] * li.opex_ratio,
        label="Operating Expenses",
    )
    ebitda = FormulaLine(
        formula=lambda li, t: li.gross_profit[t] - li.opex[t],
        label="EBITDA",
    )

In [ ]:
# Base scenario — supply the required inputs at instantiation
base = RetailModel(
    periods=[2024, 2025, 2026],
    revenue={2024: 1_000_000, 2025: 1_100_000, 2026: 1_210_000},
    opex_ratio=0.25,
)

base.tables.line_items(include_label=True, include_total_row=False).show()

In [ ]:
# Omitting a required InputLine raises a clear error
try:
    RetailModel(periods=[2024], opex_ratio=0.25)  # missing revenue
except TypeError as e:
    print(e)

### InputAssumption with a default

`InputAssumption` can have an optional default value — the caller may override it but doesn't have to.

In [ ]:
class ModelWithDefault(ProformaModel):
    growth_rate = InputAssumption(default=0.10, label="Growth Rate")
    revenue = FixedLine(values={2024: 1_000_000}, label="Revenue")
    next_revenue = FormulaLine(
        formula=lambda li, t: li.revenue[t] * (1 + li.growth_rate),
        label="Projected Revenue",
    )

# Uses the default
m_default = ModelWithDefault(periods=[2024])
print(f"Default growth_rate: {m_default.av.growth_rate}")

# Override the default
m_override = ModelWithDefault(periods=[2024], growth_rate=0.20)
print(f"Overridden growth_rate: {m_override.av.growth_rate}")

## 2. hardcoded_color — Visually Mark Input Values

In Excel financial models it's conventional to colour hardcoded inputs (often blue) so readers can distinguish them from calculated cells. `ItemRow` supports `hardcoded_color` for the same purpose.

A value is considered "input" if it comes from a `FixedLine`, `InputLine`, or a `FormulaLine` override period.

In [ ]:
from pyproforma.v2.tables import HeaderRow, ItemRow

# Show which values are hardcoded vs calculated
# Revenue (InputLine) → all blue
# COGS, gross_profit, opex, ebitda (FormulaLine) → not coloured
template = [
    HeaderRow(),
    ItemRow(name="revenue", hardcoded_color="cornflowerblue"),
    ItemRow(name="cogs"),
    ItemRow(name="gross_profit"),
    ItemRow(name="opex"),
    ItemRow(name="ebitda"),
]

base.tables.from_template(template).show()

In [ ]:
# is_input() is also queryable directly on a LineItemResult
revenue_result = base["revenue"]
cogs_result = base["cogs"]

print(f"revenue is_input(2024): {revenue_result.is_input(2024)}")  # True — InputLine
print(f"cogs is_input(2024):    {cogs_result.is_input(2024)}")     # False — FormulaLine

In [ ]:
# FormulaLine with a period override — that specific period shows as input
class MixedModel(ProformaModel):
    revenue = FixedLine(values={2024: 100, 2025: 110, 2026: 121}, label="Revenue")
    adjusted = FormulaLine(
        formula=lambda li, t: li.revenue[t] * 0.9,
        values={2025: 88},          # manual override for 2025 only
        label="Adjusted Revenue",
    )

mixed = MixedModel(periods=[2024, 2025, 2026])
adj = mixed["adjusted"]
print(f"adjusted is_input(2024): {adj.is_input(2024)}")  # False — calculated
print(f"adjusted is_input(2025): {adj.is_input(2025)}")  # True  — override
print(f"adjusted is_input(2026): {adj.is_input(2026)}")  # False — calculated

# The override period appears blue, the others don't
mixed.tables.from_template([
    HeaderRow(),
    ItemRow(name="revenue", hardcoded_color="cornflowerblue"),
    ItemRow(name="adjusted", hardcoded_color="cornflowerblue"),
]).show()

## 3. ModelComparison

`ModelComparison` places two or more model instances side-by-side. The first model is the **baseline**; all differences are computed as `compare − base`.

Models don't need to be the same class — only their shared `line_item_names` and overlapping `periods` are compared.

In [ ]:
# Build three scenarios from the same RetailModel class
base_scenario = RetailModel(
    periods=[2024, 2025, 2026],
    revenue={2024: 1_000_000, 2025: 1_100_000, 2026: 1_210_000},
    opex_ratio=0.25,
)

optimistic = RetailModel(
    periods=[2024, 2025, 2026],
    revenue={2024: 1_200_000, 2025: 1_380_000, 2026: 1_590_000},
    opex_ratio=0.22,
)

pessimistic = RetailModel(
    periods=[2024, 2025, 2026],
    revenue={2024: 850_000, 2025: 900_000, 2026: 920_000},
    opex_ratio=0.28,
)

### Two-model comparison

In [ ]:
cmp = ModelComparison(
    base_scenario,
    optimistic,
    labels=["Base", "Optimistic"],
)
print(cmp)
print(f"Common items:   {cmp.common_items}")
print(f"Common periods: {cmp.common_periods}")

In [ ]:
# Scalar difference for a single period
diff_2024 = cmp.difference("ebitda", 2024)
print(f"EBITDA difference in 2024: {diff_2024:,.0f}")

# All periods at once
diff_all = cmp.difference("ebitda")
for period, diff in diff_all.items():
    print(f"  {period}: {diff:,.0f}")

In [ ]:
# Percent difference
pct = cmp.percent_difference("ebitda", 2024)
print(f"EBITDA % difference in 2024: {pct:.1%}")

In [ ]:
# Comparison table — shows base, optimistic, and the difference row
cmp.table(
    item_names=["revenue", "gross_profit", "ebitda"],
    include_percent_difference=True,
).show()

### Three-model comparison

With three or more models the `difference()` return changes to a dict keyed by label.

In [ ]:
cmp3 = ModelComparison(
    base_scenario,
    optimistic,
    pessimistic,
    labels=["Base", "Optimistic", "Pessimistic"],
)

# Returns {label: value} for each non-base model
diffs = cmp3.difference("ebitda", 2024)
for label, diff in diffs.items():
    print(f"  {label}: {diff:+,.0f}")

In [ ]:
# Comparison table with all three scenarios
cmp3.table(
    item_names=["revenue", "ebitda"],
    include_difference=True,
).show()

### assumption_diff() — when models are the same class

If all models are instances of the same class, `assumption_diff()` surfaces the assumption values side-by-side.

In [ ]:
adiff = cmp3.assumption_diff()
for assumption, values in adiff.items():
    print(f"{assumption}:")
    for label, val in values.items():
        print(f"  {label}: {val}")

### Using .compare() as a convenience method

In [ ]:
# Identical to ModelComparison(base_scenario, optimistic, labels=[...])
cmp2 = base_scenario.compare(optimistic, labels=["Base", "Optimistic"])
print(cmp2)

### Comparing models of different classes

Only the common `line_item_names` are compared. `base_only_items` and `compare_only_items` document the structural gap.

In [ ]:
class SimpleModel(ProformaModel):
    revenue = FixedLine(values={2024: 1_000_000, 2025: 1_100_000}, label="Revenue")
    costs = FormulaLine(
        formula=lambda li, t: li.revenue[t] * 0.70, label="Total Costs"
    )
    ebitda = FormulaLine(
        formula=lambda li, t: li.revenue[t] - li.costs[t], label="EBITDA"
    )

simple = SimpleModel(periods=[2024, 2025])
detailed = RetailModel(
    periods=[2024, 2025],
    revenue={2024: 1_000_000, 2025: 1_100_000},
    opex_ratio=0.25,
)

cross_cmp = ModelComparison(simple, detailed, labels=["Simple", "Detailed"])
print(f"Common items:          {cross_cmp.common_items}")
print(f"Simple-only items:     {cross_cmp.base_only_items}")
print(f"Detailed-only items:   {cross_cmp.compare_only_items}")

cross_cmp.table(item_names=["revenue", "ebitda"]).show()